In [0]:
#Setup
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE = "/Volumes/gr5069/raw/f1_data"

pit_stops    = spark.read.csv(f"{BASE}/pit_stops.csv",          header=True, inferSchema=True)
races        = spark.read.csv(f"{BASE}/races.csv",                 header=True, inferSchema=True)
results      = spark.read.csv(f"{BASE}/results.csv",               header=True, inferSchema=True)
drivers      = spark.read.csv(f"{BASE}/drivers.csv",               header=True, inferSchema=True)
constructors = spark.read.csv(f"{BASE}/constructors.csv",          header=True, inferSchema=True)

# 1. [10 pts] What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

In [0]:
display(pit_stops)

1.1 What was the average time each driver spent at the pit stop for each race?

  **Logic:** To answer this question, I need to first groupby 'driverId' and then groupby 'raceId' and compute the "average" of the 'duration'. 
   
 

In [0]:
pit_stops_clean = pit_stops.withColumn(
    "duration_seconds",
    F.when(
        F.col("duration").contains(":"),
        F.split(F.col("duration"), ":")[0].cast("double") * 60 +
        F.split(F.col("duration"), ":")[1].cast("double")
    ).otherwise(
        F.col("duration").cast("double")
    )
)

driver_pit_stats = (
    pit_stops_clean
    .groupBy("driverId","raceId" )
    .agg(
        F.round(F.avg("duration_seconds"), 3).alias("avg_pit_duration_s")
    )
    .orderBy("driverId", "raceId")
)

display(driver_pit_stats)

**Explanation:** 
- As there is a anomolities number in row 752, whose 'duration' is '16:44.718', not double type, so I use this code to transfer the abonormal number into normal one: 
`pit_stops_clean = pit_stops.withColumn(
    "duration_seconds",
    F.when(
        F.col("duration").contains(":"),
        F.split(F.col("duration"), ":")[0].cast("double") * 60 +
        F.split(F.col("duration"), ":")[1].cast("double")
    ).otherwise(
        F.col("duration").cast("double")
    )
)`
- To better see how each driver's average pit duration in each race, I use `.orderBy("driverId", "raceId")` 

## 1.2 Provide also the slowest and fastest pit stop in each race.

**Logic:** To answer this question, I need to group the 'duration' by 'raceId' and caculate the min and max of the 'duration' to see which one is the fastest which one is the slowest.

In [0]:
race_pit_extremes = (
    pit_stops_clean
    .groupBy("raceId")
    .agg(
        F.round(F.min("duration_seconds"), 3).alias("race_fastest_pit_s"),
        F.round(F.max("duration_seconds"), 3).alias("race_slowest_pit_s"),
    )
)
display(race_pit_extremes)

**alternative routes to answer the question**

In [0]:
from pyspark.sql.window import Window

race_win = Window.partitionBy("raceId")

pit_stops_extremes = (
    pit_stops
    .withColumn("race_fastest_ms", F.min("milliseconds").over(race_win))
    .withColumn("race_slowest_ms", F.max("milliseconds").over(race_win))
    .filter(
        (F.col("milliseconds") == F.col("race_fastest_ms")) |
        (F.col("milliseconds") == F.col("race_slowest_ms"))
    )
    .withColumn("type", 
        F.when(F.col("milliseconds") == F.col("race_fastest_ms"), "fastest")
         .otherwise("slowest")
    )
    .select("raceId", "driverId", "stop", "lap", "duration", "milliseconds", "type")
    .orderBy("raceId", "type")
)

display(pit_stops_extremes)

**Explaination:**

 To better see the detail of the slowest and fastest pit for each race, I use `window `function.

# 2. [20 pts] Rank order by finishing position the average time spent at the pit stop in each race.

In [0]:
display(results)

**Logic:**
1. Join the per-driver pit-stop averages (from Q1) with `results` to get the actual `positionOrder` (finishing position) for each driver in each race.
2. Use a **window function** partitioned by `raceId` to **rank** drivers within each race by their average pit-stop duration (ascending = fastest first).
3. Display the finishing position alongside the pit-stop rank so the relationship is visible.

In [0]:
pit_vs_finish = (
    driver_pit_stats
    .join(
        results.select("raceId", "driverId", "positionOrder"),
        on=["raceId", "driverId"],
        how="inner",
    )
)
display(pit_vs_finish)

In [0]:
race_window = Window.partitionBy("raceId").orderBy(F.asc("avg_pit_duration_s"))

q2_result = (
    pit_vs_finish
    .withColumn("pit_speed_rank", F.rank().over(race_window))
    .join(races.select("raceId", "name", "year"), on="raceId", how="left")
    .join(drivers.select("driverId", "surname", "forename"), on="driverId", how="left")
    .select(
        "year", "name",
        F.concat_ws(" ", "forename", "surname").alias("driver"),
        "avg_pit_duration_s",
        "pit_speed_rank",
        F.col("positionOrder").alias("finish_position"),
    )
    .orderBy("year", "name", "pit_speed_rank")
)

display(q2_result)

# 3. [20 pts] Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

In [0]:
display(drivers)

In [0]:
# ── Q3: Impute missing driver codes ──────────────────────────────────────────

# Step 1 – Derive a candidate code from surname (first 3 chars, uppercased)
drivers_with_candidate = drivers.withColumn(
    "candidate_code",
    F.upper(F.substring(F.col("surname"), 1, 3)),
)

# Step 2 – Use existing code where available; fall back to candidate
drivers_filled = drivers_with_candidate.withColumn(
    "code_filled",
    F.when(
        F.col("code").isNotNull() & (F.trim(F.col("code")) != ""),
        F.col("code"),
    ).otherwise(F.col("candidate_code")),
)

# Step 3 – Handle collisions: if the same code_filled appears more than once,
#          append a row-number suffix to make codes unique
dedup_window = Window.partitionBy("code_filled").orderBy("driverId")

drivers_final = (
    drivers_filled
    .withColumn("code_rank", F.row_number().over(dedup_window))
    .withColumn(
        "code_unique",
        F.when(F.col("code_rank") == 1, F.col("code_filled"))
         .otherwise(F.concat(F.col("code_filled"), F.col("code_rank").cast("string"))),
    )
    .select(
        "driverId", "forename", "surname",
        F.col("code").alias("original_code"),
        "code_unique",
    )
)

# Show rows where the code was missing (i.e., we filled it in)
drivers_final.filter(F.col("original_code").isNull()).show(20, truncate=False)
display(drivers_final)

# 4. [20 pts] Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

In [0]:
# ── Q4: Youngest & Oldest driver per race ────────────────────────────────────

# Step 1 – Join results with race date and driver dob
race_drivers = (
    results
    .select("raceId", "driverId")
    .join(races.select("raceId", "name", "year", "date"), on="raceId", how="inner")
    .join(
        drivers.select("driverId", "forename", "surname", "dob"),
        on="driverId",
        how="inner",
    )
)

# Step 2 – Compute age in complete years at the time of the race
race_drivers_age = race_drivers.withColumn(
    "age",
    F.floor(F.datediff(F.col("date"), F.col("dob")) / 365.25).cast("int"),
)

# Step 3 – Window to find min/max age per race
race_win = Window.partitionBy("raceId")

race_drivers_tagged = (
    race_drivers_age
    .withColumn("min_age_in_race", F.min("age").over(race_win))
    .withColumn("max_age_in_race", F.max("age").over(race_win))
    .withColumn(
        "youngest_in_race",
        F.when(F.col("age") == F.col("min_age_in_race"), F.lit(True)).otherwise(F.lit(False)),
    )
    .withColumn(
        "oldest_in_race",
        F.when(F.col("age") == F.col("max_age_in_race"), F.lit(True)).otherwise(F.lit(False)),
    )
)

# Step 4 – Display only the youngest and oldest per race
q4_result = (
    race_drivers_tagged
    .filter(F.col("youngest_in_race") | F.col("oldest_in_race"))
    .select(
        "year", "name",
        F.concat_ws(" ", "forename", "surname").alias("driver"),
        "dob", "date", "age",
        "youngest_in_race", "oldest_in_race",
    )
    .orderBy("year", "name", F.desc("oldest_in_race"))
)

display(q4_result)

# 5. [20 pts] At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

In [0]:
# ── Q5: Cumulative podiums per driver at each race ────────────────────────────

# Step 1 – Indicator columns for each podium position
results_with_flags = (
    results
    .filter(F.col("positionOrder").isin([1, 2, 3]))  # only classified finishers on podium
    .withColumn("is_win",    F.when(F.col("positionOrder") == 1, 1).otherwise(0))
    .withColumn("is_p2",     F.when(F.col("positionOrder") == 2, 1).otherwise(0))
    .withColumn("is_p3",     F.when(F.col("positionOrder") == 3, 1).otherwise(0))
)

# We need ALL races for each driver (not just podium ones) to show 0s too
all_results_flags = (
    results
    .withColumn("is_win", F.when(F.col("positionOrder") == 1, 1).otherwise(0))
    .withColumn("is_p2",  F.when(F.col("positionOrder") == 2, 1).otherwise(0))
    .withColumn("is_p3",  F.when(F.col("positionOrder") == 3, 1).otherwise(0))
)

# Step 2 – Cumulative window per driver ordered by raceId (proxy for chronology)
cum_win = (
    Window
    .partitionBy("driverId")
    .orderBy("raceId")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

q5_result = (
    all_results_flags
    .withColumn("cum_wins", F.sum("is_win").over(cum_win))
    .withColumn("cum_p2",   F.sum("is_p2").over(cum_win))
    .withColumn("cum_p3",   F.sum("is_p3").over(cum_win))
    .withColumn("total_podiums", F.col("cum_wins") + F.col("cum_p2") + F.col("cum_p3"))
    .join(races.select("raceId", "name", "year"), on="raceId", how="left")
    .join(
        drivers.select("driverId", "forename", "surname"),
        on="driverId",
        how="left",
    )
    .select(
        "year", "name",
        F.concat_ws(" ", "forename", "surname").alias("driver"),
        "positionOrder",
        "cum_wins", "cum_p2", "cum_p3", "total_podiums",
    )
    .orderBy("year", "name", F.desc("total_podiums"))
)

display(q5_result)

# 6. [10 pts] Continue exploring the data by answering your own question.

### Question
**Which constructor (team) has the best (lowest) average finishing position per season, and how has this ranking shifted year-over-year?**


In [0]:
# ── Q6: Best constructor average finishing position by season ─────────────────

# Step 1 – Join and filter classified finishes
constructor_results = (
    results
    .filter(F.col("positionOrder") <= 20)   # exclude non-classified (DNF/DSQ coded as 0 or >20)
    .join(races.select("raceId", "year"), on="raceId", how="inner")
    .join(
        constructors.select("constructorId", F.col("name").alias("constructor")),
        on="constructorId",
        how="inner",
    )
)

# Step 2 – Average finishing position per constructor per season
constructor_avg = (
    constructor_results
    .groupBy("year", "constructorId", "constructor")
    .agg(F.round(F.avg("positionOrder"), 2).alias("avg_finish_pos"))
)

# Step 3 – Rank within each year (1 = best average finishing position)
year_win = Window.partitionBy("year").orderBy(F.asc("avg_finish_pos"))

q6_result = (
    constructor_avg
    .withColumn("season_rank", F.rank().over(year_win))
    .filter(F.col("season_rank") <= 5)      # top 5 constructors per season
    .orderBy("year", "season_rank")
    .select("year", "season_rank", "constructor", "avg_finish_pos")
)

display(q6_result)